# Chapter 3: Gaussian Filters
## KF / EKF / UKF / IF — Side-by-side Comparison

All four filters on the same figure-8 trajectory with landmark observations.
**Key insight**: UKF typically matches or beats EKF without requiring Jacobian derivation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt

from pluto_filters.pluto_filters.kalman_filters.ekf import EKF, LANDMARKS, normalize_angle
from pluto_filters.pluto_filters.kalman_filters.ukf import UKF
from pluto_experiments.pluto_experiments.filter_showdown.benchmark import (
    generate_figure8_trajectory, generate_measurements, rmse, nees
)

In [ ]:
np.random.seed(42)
true_poses, dt = generate_figure8_trajectory(300, 0.1)
measurements = generate_measurements(true_poses, landmark_id=0)

def run_filter(filter_cls, v=0.3, omega=0.2):
    mu0 = np.array([0.3, 0.2, 0.15])
    sigma0 = np.diag([1.0, 1.0, 0.2])
    f = filter_cls(mu0, sigma0)
    ests = []
    for i, (r, phi) in enumerate(measurements):
        if i > 0:
            f.predict(v, omega, dt)
        f.update(0, r, phi)
        ests.append(f.mu.copy())
    return np.array(ests)

ekf_ests = run_filter(EKF)
ukf_ests = run_filter(UKF)

print(f'EKF RMSE: {rmse(ekf_ests, true_poses):.4f} m')
print(f'UKF RMSE: {rmse(ukf_ests, true_poses):.4f} m')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (ests, name, color) in zip(axes, [
    (ekf_ests, 'EKF', 'steelblue'),
    (ukf_ests, 'UKF', 'coral'),
]):
    ax.plot(true_poses[:, 0], true_poses[:, 1], 'k-', lw=2, label='Ground Truth')
    ax.plot(ests[:, 0], ests[:, 1], '--', color=color, lw=1.5, label=name)
    for lm_id, (lx, ly) in LANDMARKS.items():
        ax.plot(lx, ly, 'k^', markersize=10)
        ax.annotate(f'LM{lm_id}', (lx, ly), textcoords='offset points', xytext=(5, 5))
    ax.set_title(f'{name} — RMSE={rmse(ests, true_poses):.4f}m')
    ax.legend()
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')

plt.suptitle('Figure-8 Trajectory: EKF vs UKF', fontsize=14)
plt.tight_layout()
plt.savefig('ch03_gaussian_filters.png', dpi=120)
plt.show()

## Exercises
1. Increase `sigma0` to 10.0 — which filter takes longer to converge?
2. Remove 3 of the 5 landmarks — how does RMSE change?
3. The Information Filter: implement a comparison showing how Ω grows as measurements arrive.